In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [2]:
import pandas as pd
res = pd.read_csv("reports/summary_metrics.csv")
(res["MaxDD"] < -1).sum(), res["MaxDD"].min()

(np.int64(0), np.float64(-0.447990053355547))

In [3]:
import src.run as run
print(run.__file__)

/Users/luchengliang/Quant-regime-project/src/run.py


In [4]:
import inspect
from src.run import perf_metrics

print(inspect.getsource(perf_metrics)[:400])

def perf_metrics(strat_ret: pd.Series, periods: int = 252):
    r = strat_ret.copy()
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0.0)
    eq = equity_from_returns(r)
    T = len(r)
    if T == 0:
        metrics = dict(FinalEquity=np.nan, AnnRet=np.nan, AnnVol=np.nan, Sharpe=np.nan, MaxDD=np.nan, Calmar=np.nan)
        return metrics, eq, pd.Series(dtype=float)

    final_eq = float(eq.ilo


In [5]:
import itertools
import pandas as pd

from src.run import Config, run_symbol

grid = {
    "mr_window": [10, 20, 30, 50],
    "vol_q": [0.6, 0.7, 0.8],
    "fee_bps": [0.5, 1.0, 2.0],
}
symbols = ["SPY", "2330.TW"]
base_cfg = Config()

keys = list(grid.keys())
combos = list(itertools.product(*[grid[k] for k in keys]))

rows = []
for values in combos:
    cfg_kwargs = base_cfg.__dict__.copy()
    for k, v in zip(keys, values):
        cfg_kwargs[k] = v
    cfg = Config(**cfg_kwargs)

    for sym in symbols:
        table, _, _ = run_symbol(sym, cfg)
        for _, r in table.iterrows():
            rec = r.to_dict()
            for k, v in zip(keys, values):
                rec[f"param_{k}"] = v
            rows.append(rec)

res = pd.DataFrame(rows)
print((res["MaxDD"] < -1).sum(), res["MaxDD"].min())

res.to_csv("reports/grid_results.csv", index=False)
res.head()

0 -0.447990053355547


,Symbol,Strategy,FinalEquity,AnnRet,AnnVol,Sharpe,MaxDD,Calmar,param_mr_window,param_vol_q,param_fee_bps
0,SPY,Buy&Hold,6.572025,0.136310,0.170940,0.833541,-0.337172,0.404275,10,0.6,0.5
1,SPY,Momentum,0.951758,-0.003350,0.090008,0.008328,-0.397668,-0.008424,10,0.6,0.5
2,SPY,Mean Reversion,2.470379,0.063302,0.135141,0.522099,-0.382131,0.165656,10,0.6,0.5
3,SPY,Mean Reversion with Regime,1.936700,0.045882,0.074120,0.642380,-0.170304,0.269414,10,0.6,0.5
4,2330.TW,Buy&Hold,27.586594,0.262697,0.252545,1.049983,-0.447990,0.586390,10,0.6,0.5
